# Project 20 — BROKEN notebook (debugging exercise)

Seeded bugs centred on the capstone pitfalls: **ranking by raw (unpooled) means** (the winner's curse) and **stopping at the posterior** without a loss function. Run it, see the wrong recommendation, find each bug, fix it. Answer key: `BROKEN_BUGS.md`.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
RNG = 20240601

In [ ]:
from data.generate_data import generate
data = generate()

### BUG 1 — rank compounds by their RAW means (no pooling).

This ignores that low-replicate compounds have noisy means; the 'winner' is often a small-$n$ fluke.

In [ ]:
# BUG 1: pick the compound with the largest raw mean.
raw_pick = int(np.argmax(data['raw_means']))
print(f"raw-mean recommendation = #{raw_pick} "
      f"(n={data['n_reps'][raw_pick]}, raw={data['raw_means'][raw_pick]:.2f})")
print(f"but the TRUE best is #{data['best_true']} "
      f"(theta={data['theta_true'][data['best_true']]:.2f})")

### BUG 2 — fit a model but STOP at the posterior mean.

Even a correct hierarchical fit is not a decision. Ranking by posterior mean ignores uncertainty (probability-of-best, expected regret). And here a second seeded modelling bug lurks: a CENTRED hierarchy that may diverge.

In [ ]:
# BUG 3: CENTRED hierarchy -> funnel / divergences at small tau.
comp_idx = data['comp_idx']; y = data['y']; J = data['j']
with pm.Model() as model:
    mu = pm.Normal('mu', 0, 2)
    tau = pm.HalfNormal('tau', 1)
    theta = pm.Normal('theta', mu=mu, sigma=tau, shape=J)  # CENTRED
    sigma = pm.HalfNormal('sigma', 1)
    pm.Normal('y_obs', mu=theta[comp_idx], sigma=sigma, observed=y)
    idata = pm.sample(draws=500, tune=800, chains=2, cores=1,
                      target_accept=0.9, random_seed=RNG, progressbar=False)
print('divergences:', int(idata.sample_stats['diverging'].sum()))

In [ ]:
# BUG 2 continued: stopping at the posterior mean, no loss function.
pm_pick = int(idata.posterior['theta'].mean(dim=('chain','draw')).argmax())
print(f"posterior-mean recommendation = #{pm_pick} (no decision-theoretic step)")
# The fix: compute expected utility / expected regret / P(best) and recommend
# the min-regret compound -- see model.decision_table and the clean notebook.